In [1]:
from datasets import load_dataset
for cfg in ["corpus", "queries", "default"]:
    ds = load_dataset("mteb/fiqa", cfg)
    print(cfg, {k: (v.num_rows, v.column_names) for k, v in ds.items()})

C:\Users\danim\AppData\Roaming\Python\Python314\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\danim\AppData\Roaming\Python\Python314\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\danim\.cache\huggingface\hub\datasets--mteb--fiqa. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activat

corpus {'corpus': (57638, ['_id', 'title', 'text'])}


Generating queries split: 100%|██████████| 6648/6648 [00:00<00:00, 155463.25 examples/s]


queries {'queries': (6648, ['_id', 'text'])}


Generating test split: 100%|██████████| 1706/1706 [00:00<00:00, 52429.57 examples/s]

default {'train': (14166, ['query-id', 'corpus-id', 'score']), 'dev': (1238, ['query-id', 'corpus-id', 'score']), 'test': (1706, ['query-id', 'corpus-id', 'score'])}


In [3]:
!python -m scripts.crear_indice
!python -m scripts.crear_embeddings

Corpus: 20000 docs | Consultas: 648 | Pares relevantes: 1706
Términos únicos: 31080 | Longitud promedio: 71.2 tokens
Listo en 21.3 s


Matriz de embeddings: (20000, 384)
Vectores en FAISS: 20000 | Listo en 1337.8 s



Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
C:\Users\danim\AppData\Roaming\Python\Python314\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\DANIMF\WORKCENTER\SEPTIMO_SEMESTRE_1\RECUPERACION_INFORMACION\PROYECTO1B\Proyecto_Recuperacion_de_Informacion\artifacts\modelo_cache. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to act

In [5]:
import config
from src import almacen_vectores
from src.data_loader import cargar_json
from src.indice_invertido import IndiceInvertido
from src.modelos.modelo_binario import ModeloJaccard
from src.modelos.tfidf_coseno import ModeloTfidfCoseno
from src.modelos.bm25 import ModeloBM25
from src.modelos.semantico import GeneradorEmbeddings, ModeloSemantico

indice = IndiceInvertido.cargar(config.RUTA_INDICE_INVERTIDO)
faiss_idx, doc_ids = almacen_vectores.cargar(config.RUTA_FAISS, config.RUTA_DOC_IDS)
modelos = [ModeloJaccard(indice), ModeloTfidfCoseno(indice), ModeloBM25(indice),
           ModeloSemantico(GeneradorEmbeddings(), faiss_idx, doc_ids)]

consultas, qrels = cargar_json(config.RUTA_CONSULTAS), cargar_json(config.RUTA_QRELS)
qid = next(iter(consultas))
print(consultas[qid], "| relevantes:", list(qrels[qid]))
for m in modelos:
    print(m.nombre, [d for d, _ in m.buscar(consultas[qid], 5)])

Where should I park my rainy-day / emergency fund? | relevantes: ['44594', '406219', '319954', '397358', '88327']
Binario (Jaccard) ['290830', '435407', '406219', '569528', '372677']
Vectorial (TF-IDF) ['551764', '282623', '538023', '284318', '62162']
Probabilístico (BM25) ['282623', '538023', '284318', '214248', '588555']
Vectorial (Embeddings) ['44594', '406219', '88327', '285812', '553748']


## VALIDACIONES

### 1. Carga de artefactos e integridad del corpus

Se verifica que los archivos generados en artifacts/ existan, no estén vacíos y que los IDs de documentos coincidan con los qrels

In [4]:
import os
import numpy as np
import config
from src.data_loader import cargar_json
from src.indice_invertido import IndiceInvertido

# Cargar artefactos
indice = IndiceInvertido.cargar(config.RUTA_INDICE_INVERTIDO)
consultas = cargar_json(config.RUTA_CONSULTAS)
qrels = cargar_json(config.RUTA_QRELS)
doc_ids = cargar_json(config.RUTA_DOC_IDS)

# Ajusta el nombre si en config.py se llama distinto
ruta_doc_store = getattr(config, "RUTA_DOC_STORE", os.path.join("artifacts", "doc_store.json"))
doc_store = cargar_json(ruta_doc_store)

assert len(doc_store) > 0, "doc_store está vacío"
assert len(consultas) > 0, "consultas está vacío"
assert len(qrels) > 0, "qrels está vacío"
assert len(doc_ids) == len(doc_store), f"Desalineación: doc_ids ({len(doc_ids)}) != doc_store ({len(doc_store)})"

# Verificar que los documentos relevantes en qrels realmente existan en el corpus
doc_set = set(doc_ids)
faltantes = [did for rels in qrels.values() for did in rels if did not in doc_set]
print(f"Documentos en corpus: {len(doc_store)}")
print(f"Consultas totales: {len(consultas)} | Consultas con qrels: {len(qrels)}")
print(f"Docs relevantes en qrels ausentes en el corpus: {len(faltantes)}")
assert len(faltantes) == 0, "Hay documentos en qrels que no existen en tu corpus/doc_ids"

Documentos en corpus: 20000
Consultas totales: 648 | Consultas con qrels: 648
Docs relevantes en qrels ausentes en el corpus: 0


### 2. Preprocesamiento (Sección a: tokenización, normalización, stopwords)

Se valida que el módulo src/preprocesamiento.py elimine puntuación, convierta a minúsculas, remueva stopwords y no falle con entradas vacías.

In [6]:
import src.preprocesamiento as prep

# Cambia 'preprocesar' por el nombre exacto de tu función si se llama distinto
fn_prep = getattr(prep, "preprocesar", getattr(prep, "tokenizar", None))
assert fn_prep is not None, f"Define la función correcta. Disponibles: {[f for f in dir(prep) if not f.startswith('_')]}"

casos = {
    "normal": "How do I invest in Index Funds, ETFs, and Stocks in 2026?!",
    "solo_stopwords": "the and of to in a is for on with",
    "vacio": "",
    "simbolos": "!!! ??? --- $$$ 12345"
}

for nombre, texto in casos.items():
    tokens = fn_prep(texto)
    print(f"[{nombre}] -> {tokens}")
    assert isinstance(tokens, list), "El preprocesamiento debe devolver una lista de tokens"
    assert all(t == t.lower() for t in tokens), "Quedaron tokens en mayúsculas"

assert len(fn_prep(casos["vacio"])) == 0, "Un texto vacío debe devolver lista vacía"
assert len(fn_prep(casos["solo_stopwords"])) == 0, "No se están removiendo las stopwords"
print("OK: Preprocesamiento validado.")

[normal] -> ['invest', 'index', 'fund', 'etf', 'stock', '2026']
[solo_stopwords] -> []
[vacio] -> []
[simbolos] -> ['12345']
OK: Preprocesamiento validado.


### 3. Integridad del Índice Invertido (Sección a)

Se comprueba que el índice almacene términos, documentos y frecuencias coherentes.

In [8]:
# 1. Inspeccionar cómo está guardado un elemento en doc_store
primer_id = next(iter(doc_store))
print("Tipo y contenido del primer registro:", type(doc_store[primer_id]), doc_store[primer_id])

def extraer_texto(doc):
    if isinstance(doc, str):
        return doc
    if isinstance(doc, dict):
        # Une todos los valores de texto del diccionario sin depender del nombre de la clave
        return " ".join(str(v) for v in doc.values() if isinstance(v, str))
    return ""

# 2. Buscar el primer documento que tenga tokens reales tras preprocesar
sample_doc_id, tokens_sample = None, []
for did, contenido in doc_store.items():
    toks = fn_prep(extraer_texto(contenido))
    if len(toks) > 0:
        sample_doc_id, tokens_sample = did, toks
        break

assert sample_doc_id is not None, "Ningún documento del corpus produjo tokens. Revisa preprocesamiento.py o doc_store.json"
print(f"Doc válido encontrado: '{sample_doc_id}' con {len(tokens_sample)} tokens.")

# 3. Verificar en indice.postings
termino_prueba = tokens_sample[0]
assert termino_prueba in indice.postings, f"El término '{termino_prueba}' no está en indice.postings"

postings = indice.postings[termino_prueba]
print(f"Término '{termino_prueba}' aparece en {len(postings)} documentos.")
assert sample_doc_id in postings, f"El doc {sample_doc_id} no aparece en postings['{termino_prueba}']"

if isinstance(postings, dict):
    assert all(tf > 0 for tf in postings.values()), "Existen frecuencias <= 0 en el índice invertido"

assert indice.n_docs == len(doc_store), f"n_docs ({indice.n_docs}) != len(doc_store) ({len(doc_store)})"
print("OK: Índice invertido consistente.")

Tipo y contenido del primer registro: <class 'dict'> {'titulo': '', 'texto': '"This is the best tl;dr I could make, [original](http://www.aljazeera.com/news/2017/10/israel-engages-water-apartheid-171013110734930.html) reduced by 91%. (I\'m a bot) ***** &gt; &amp;quot;In reality, Israel doesn&amp;#039;t want us to live here any more We just want the Israelis to let us extract our own water.""  &gt; Mekorot, Israel&amp;#039;s national water company, dug deep into the mountain aquifer, and by the end of the 1970s, Israel had extracted so much water that the springs in Bardala and Ein al-Beida had dried up.  &gt; &amp;quot;The level of unrestricted access to water enjoyed by those residing in Israel and Israeli settlers demonstrates that resources are plentiful, and that the lack of sufficient water for Palestinians is a direct result of Israel&amp;#039;s discriminatory policies in water management.""   ***** [**Extended Summary**](http://np.reddit.com/r/autotldr/comments/77x7fl/how_israel

### 4. Modelos Clásicos — Reglas matemáticas y casos borde (Sección b)
Valida Jaccard, TF-IDF Coseno y BM25: ordenamiento descendente, ausencia de duplicados, rangos válidos de puntuación y manejo de consultas vacías o fuera de vocabulario.

In [9]:
from src.modelos.modelo_binario import ModeloJaccard
from src.modelos.tfidf_coseno import ModeloTfidfCoseno
from src.modelos.bm25 import ModeloBM25

modelos_clasicos = [
    ModeloJaccard(indice),
    ModeloTfidfCoseno(indice),
    ModeloBM25(indice),
]

q_libre = "capital gains tax on stock dividends"
k = 10

for m in modelos_clasicos:
    resultados = m.buscar(q_libre, k)
    assert len(resultados) <= k, f"{m.nombre} devolvió más de {k} resultados"
    
    docs = [d for d, _ in resultados]
    scores = [s for _, s in resultados]
    
    # 1. Sin duplicados
    assert len(docs) == len(set(docs)), f"{m.nombre} devolvió documentos duplicados"
    # 2. Orden descendente estricto
    assert scores == sorted(scores, reverse=True), f"{m.nombre} no ordena descendentemente"
    # 3. Rangos matemáticos
    nombre_lower = m.nombre.lower()
    if "jaccard" in nombre_lower or "binario" in nombre_lower:
        assert all(0.0 <= s <= 1.0001 for s in scores), f"Jaccard fuera de [0,1]: {scores[:3]}"
    elif "tfidf" in nombre_lower or "coseno" in nombre_lower:
        assert all(0.0 <= s <= 1.0001 for s in scores), f"Coseno fuera de [0,1]: {scores[:3]}"
    elif "bm25" in nombre_lower:
        assert all(s >= 0.0 for s in scores), f"BM25 con score negativo: {scores[:3]}"

    # 4. Casos borde (no deben lanzar ZeroDivisionError ni crashear)
    for borde in ["", "the and of in", "xyzqwerty999noexiste"]:
        res_borde = m.buscar(borde, 5)
        assert res_borde == [], f"{m.nombre} devolvió resultados para caso borde '{borde}': {res_borde}"

    print(f"OK [{m.nombre}] -> Top-3 scores: {[round(s, 4) for s in scores[:3]]}")

OK [Binario (Jaccard)] -> Top-3 scores: [0.2857, 0.2857, 0.2727]
OK [Vectorial (TF-IDF)] -> Top-3 scores: [0.4764, 0.4742, 0.4739]
OK [Probabilístico (BM25)] -> Top-3 scores: [23.1357, 22.8625, 22.1822]


### 5. Contraste de rankings y acierto sobre qrels (Sección b)
Confirma que los tres modelos producen rankings diferenciados (descarta bugs de copiar/pegar fórmulas) y que recuperan documentos relevantes en una muestra de consultas reales.

In [10]:
muestra_qids = list(qrels.keys())[:5]

for qid in muestra_qids:
    texto_q = consultas[qid]
    relevantes = set(qrels[qid])
    print(f"\nQID {qid}: {texto_q[:80]}... (Relevantes: {len(relevantes)})")
    
    rankings = {}
    for m in modelos_clasicos:
        top10 = [d for d, _ in m.buscar(texto_q, 10)]
        rankings[m.nombre] = top10
        hits = len(set(top10) & relevantes)
        print(f"  {m.nombre:20s} | Hits@10: {hits}/{min(10, len(relevantes))} | Top-3: {top10[:3]}")


QID 8: How to deposit a cheque issued to an associate in my business into my business a... (Relevantes: 2)
  Binario (Jaccard)    | Hits@10: 1/2 | Top-3: ['188893', '273989', '41484']
  Vectorial (TF-IDF)   | Hits@10: 1/2 | Top-3: ['65404', '220644', '319786']
  Probabilístico (BM25) | Hits@10: 1/2 | Top-3: ['65404', '212713', '319786']

QID 15: Can I send a money order from USPS as a business?... (Relevantes: 1)
  Binario (Jaccard)    | Hits@10: 0/1 | Top-3: ['397847', '252097', '330223']
  Vectorial (TF-IDF)   | Hits@10: 0/1 | Top-3: ['321826', '373199', '468361']
  Probabilístico (BM25) | Hits@10: 0/1 | Top-3: ['321826', '26979', '582933']

QID 18: 1 EIN doing business under multiple business names... (Relevantes: 1)
  Binario (Jaccard)    | Hits@10: 0/1 | Top-3: ['188893', '594175', '27273']
  Vectorial (TF-IDF)   | Hits@10: 1/1 | Top-3: ['140097', '504317', '348480']
  Probabilístico (BM25) | Hits@10: 1/1 | Top-3: ['348480', '504317', '431685']

QID 26: Applying for and receiving

### 6. Embeddings de consultas de texto libre y sanidad semántica (Sección c)
Valida GeneradorEmbeddings: dimensiones, ausencia de NaN/Inf, norma L2 (importante si usas producto interno en FAISS) y que dos consultas con el mismo significado pero distintas palabras tengan mayor similitud coseno que una consulta de otro tema.

In [12]:
from src.modelos.semantico import GeneradorEmbeddings

gen = GeneradorEmbeddings()

def a_vector_1d(texto):
    vec = gen.embeber_consulta(texto)
    return np.asarray(vec, dtype=np.float32).squeeze()

def coseno(u, v):
    return float(np.dot(u, v) / (np.linalg.norm(u) * np.linalg.norm(v) + 1e-12))

q_base = "How to reduce taxes when selling stocks?"
q_parafrasis = "Legal ways to pay less capital gains on equity investments"
q_no_relacionada = "Best recipe for homemade chocolate cake with strawberries"

v_base = a_vector_1d(q_base)
v_para = a_vector_1d(q_parafrasis)
v_irrel = a_vector_1d(q_no_relacionada)

# Verificar compatibilidad de dimensiones entre consultas y documentos
v_docs = np.asarray(gen.embeber_documentos(["Sample financial document text"]), dtype=np.float32)

assert v_base.ndim == 1, f"Se esperaba vector 1D, se obtuvo shape {v_base.shape}"
assert v_base.shape == v_para.shape == v_irrel.shape, "Dimensiones inconsistentes entre consultas"
assert v_docs.shape[-1] == v_base.shape[0], f"Dimensión docs ({v_docs.shape[-1]}) != consulta ({v_base.shape[0]})"
assert not np.isnan(v_base).any() and not np.isinf(v_base).any(), "El embedding contiene NaN o Inf"

norma = float(np.linalg.norm(v_base))
sim_relacionada = coseno(v_base, v_para)
sim_irrelacionada = coseno(v_base, v_irrel)

print(f"Dimensión del embedding: {v_base.shape[0]}")
print(f"Norma L2 del vector: {norma:.4f} (si es ~1.0, está normalizado)")
print(f"Similitud (paráfrasis):     {sim_relacionada:.4f}")
print(f"Similitud (no relacionada): {sim_irrelacionada:.4f}")

assert sim_relacionada > sim_irrelacionada + 0.15, "Fallo semántico: el modelo no distingue bien entre paráfrasis y tema ajeno"
print("OK: Generación de embeddings para consultas y documentos validada.")

Embeddings: 100%|██████████| 1/1 [00:00<00:00, 166.73it/s]

Dimensión del embedding: 384
Norma L2 del vector: 1.0000 (si es ~1.0, está normalizado)
Similitud (paráfrasis):     0.5077
Similitud (no relacionada): 0.0973
OK: Generación de embeddings para consultas y documentos validada.


### 7. Consistencia previa a la base de datos vectorial (Puente hacia FAISS)
Esta celda confirma que el índice vectorial tiene la misma dimensión que tu generador de embeddings y la misma cantidad de documentos que doc_ids.

In [13]:
from src import almacen_vectores

faiss_idx, doc_ids_faiss = almacen_vectores.cargar(config.RUTA_FAISS, config.RUTA_DOC_IDS)

print(f"Vectores en FAISS: {faiss_idx.ntotal} | doc_ids: {len(doc_ids_faiss)} | Dimensión FAISS: {faiss_idx.d}")
assert faiss_idx.ntotal == len(doc_ids_faiss) == len(doc_store), "Cantidad de vectores en FAISS no coincide con el corpus"
assert faiss_idx.d == v_base.shape[0], f"Dimensión FAISS ({faiss_idx.d}) != dimensión embedding ({v_base.shape[0]})"
print("OK: Consistencia entre embeddings, doc_ids y FAISS verificada.")

Vectores en FAISS: 20000 | doc_ids: 20000 | Dimensión FAISS: 384
OK: Consistencia entre embeddings, doc_ids y FAISS verificada.


In [14]:
from src.modelos.semantico import ModeloSemantico

m_sem = ModeloSemantico(gen, faiss_idx, doc_ids_faiss)
res_sem = m_sem.buscar("capital gains tax on stock dividends", 10)

docs_sem = [d for d, _ in res_sem]
scores_sem = [s for _, s in res_sem]

assert len(res_sem) == 10, f"Se esperaban 10 resultados, devolvió {len(res_sem)}"
assert len(docs_sem) == len(set(docs_sem)), "ModeloSemantico devolvió documentos duplicados"
assert scores_sem == sorted(scores_sem, reverse=True), "ModeloSemantico no ordena descendentemente"
assert m_sem.buscar("", 5) == [] or isinstance(m_sem.buscar("", 5), list), "Fallo con consulta vacía"

print(f"OK [{m_sem.nombre}] -> Top-3: {[(d, round(s, 4)) for d, s in res_sem[:3]]}")

OK [Vectorial (Embeddings)] -> Top-3: [('321114', 0.7171), ('42521', 0.6682), ('295082', 0.667)]


### Seccion e y f


In [16]:
!python -m scripts.evaluacion

Cargando artefactos...

Evaluando 4 modelos sobre 648 consultas (k=10)...

Modelo                    | MAP@10   | P@10     | R@10     | Latencia (ms)
--------------------------------------------------------------------------
Binario (Jaccard)         | 0.0478   | 0.0258   | 0.1064   | 7.63         
Vectorial (TF-IDF)        | 0.1868   | 0.0755   | 0.3328   | 12.70        
Probabilístico (BM25)     | 0.2640   | 0.0926   | 0.4134   | 12.74        
Vectorial (Embeddings)    | 0.3884   | 0.1309   | 0.5455   | 17.73        

Métricas exportadas a: C:\DANIMF\WORKCENTER\SEPTIMO_SEMESTRE_1\RECUPERACION_INFORMACION\PROYECTO1B\Proyecto_Recuperacion_de_Informacion\artifacts\metricas.json
Casos comparativos exportados a: C:\DANIMF\WORKCENTER\SEPTIMO_SEMESTRE_1\RECUPERACION_INFORMACION\PROYECTO1B\Proyecto_Recuperacion_de_Informacion\artifacts\analisis_comparativo.json



#### 1. Comparación Cuantitativa Global ($k = 10$, 648 consultas)

| Modelo | MAP@10 | Precision@10 | Recall@10 | Latencia promedio (ms) |
| :--- | :---: | :---: | :---: | :---: |
| **Binario (Jaccard)** | 0.0478 | 0.0258 | 0.1064 | 7.63 |
| **Vectorial (TF-IDF)** | 0.1868 | 0.0755 | 0.3328 | 12.70 |
| **Probabilístico (BM25)** | 0.2640 | 0.0926 | 0.4134 | 12.74 |
| **Vectorial (Embeddings)** | **0.3884** | **0.1309** | **0.5455** | 17.73 |

* **Binario (Jaccard):** Presenta el rendimiento más bajo ($\text{MAP@10} = 0.0478$) porque trata todos los términos con igual peso (presencia/ausencia). Palabras frecuentes con bajo poder discriminativo penalizan la intersección frente a documentos largos.
* **Vectorial (TF-IDF):** Mejora sustancialmente frente a Jaccard ($\text{MAP@10} = 0.1868$) al ponderar la rareza de los términos mediante IDF y normalizar por la longitud del vector documento.
* **Probabilístico (BM25):** Supera a TF-IDF en un $+41.3\%$ de MAP ($0.2640$) gracias a la saturación de frecuencia de término ($k_1$) y la penalización no lineal por longitud de documento ($b$), evitando que documentos muy extensos dominen el ranking por repetición léxica.
* **Vectorial (Embeddings - `all-MiniLM-L6-v2`):** Obtiene el mejor desempeño global ($\text{MAP@10} = 0.3884$, $+47.1\%$ sobre BM25), recuperando más de la mitad de los documentos relevantes en el Top-10 ($\text{Recall@10} = 0.5455$) con un costo computacional bajo ($17.73\text{ ms}$ por consulta en FAISS).

#### 2. Tipos de consultas donde destaca cada modelo

* **Jaccard:** Solo resulta competitivo en consultas cortas con palabras clave altamente específicas donde el documento relevante tiene una longitud similar a la consulta (ej. `qid: 6410`, $\text{AP} = 1.0$).
* **TF-IDF y BM25:** Funcionan mejor en consultas que contienen **siglas, códigos financieros, nombres propios o terminología técnica exacta** (ej. `403b`, `401k`, `IOPTS`, `pre-market`), donde la coincidencia léxica directa es el indicador principal de relevancia. BM25 destaca especialmente cuando los documentos relevantes son respuestas extensas de foro con alta variabilidad de longitud.
* **Vectorial con Embeddings:** Destaca en **preguntas formuladas en lenguaje natural, consultas conceptuales o de intención** donde la respuesta correcta utiliza paráfrasis, ejemplos numéricos o vocabulario distinto al de la pregunta.

#### 3. Casos donde la recuperación semántica mejora frente a BM25 y TF-IDF

1. **Paráfrasis y ausencia de vocabulario compartido (`qid: 106`):**
   * *Consulta:* `"What approaches are there for pricing a small business?"`
   * *Resultados:* Embeddings obtiene $\text{AP} = 1.0$ (recuperando `doc_id: 76695` en posición 1), mientras que BM25 y TF-IDF obtienen $\text{AP} = 0.0$. El documento relevante habla de *"liquidation value"* y valoración de un *"B&B"* sin repetir la palabra *"approaches"*.
2. **Consultas con razonamiento práctico o numérico (`qid: 604`):**
   * *Consulta:* `"Is there a dollar amount that, when adding Massachusetts Sales Tax, precisely equals $200?"`
   * *Resultados:* Embeddings recupera los 3 documentos relevantes en las primeras 3 posiciones ($\text{AP} = 1.0$, $\text{Recall@10} = 1.0$), mientras que TF-IDF y BM25 fallan ($\text{AP} = 0.0$). Los documentos relevantes contienen el cálculo matemático (`$188.24`, `0.0625x`, `$200.01`) en lugar de repetir el nombre del estado (*Massachusetts*).
3. **Consultas operativas en lenguaje cotidiano (`qid: 15` y `qid: 1824`):**
   * *Consultas:* `"Can I send a money order from USPS as a business?"` (`qid: 15`) y `"Is there a way to open a U.S. bank account for my LLC remotely?"` (`qid: 1824`).
   * *Resultados:* En ambos casos Embeddings logra $\text{AP} = 1.0$ frente a $\text{AP} = 0.0$ de los modelos léxicos, capturando la equivalencia semántica entre *"remotely"* y *"completely online signup"* (`doc_id: 244808`).

#### 4. Casos donde la recuperación semántica empeora frente a BM25 y TF-IDF

1. **Identificadores y códigos financieros específicos (`qid: 9961` y `qid: 9329`):**
   * *Consultas:* `"Employer rollover from 403b to 401k?"` (`qid: 9961`) e `"Interactive Brokers: IOPTS and list of structured products"` (`qid: 9329`).
   * *Resultados:* En `qid: 9961`, BM25 y TF-IDF alcanzan $\text{AP} = 1.0$ (`doc_id: 21311`), mientras Embeddings cae a $\text{AP} = 0.0$. En `qid: 9329`, TF-IDF obtiene $\text{AP} = 1.0$ y BM25 $\text{AP} = 0.7$, mientras Embeddings obtiene $\text{AP} = 0.0$. Los embeddings densos tienden a mapear siglas poco frecuentes (`403b`, `IOPTS`) hacia el vecindario genérico de "cuentas de retiro" o "instrumentos financieros", perdiendo la exactitud del identificador.
2. **Términos compuestos de dominio específico (`qid: 9556` y `qid: 6410`):**
   * *Consultas:* `"How does pre-market trading work?"` (`qid: 9556`, BM25 $\text{AP} = 1.0$ vs. Embeddings $\text{AP} = 0.0$) y `"Will an ETF immediately reflect a reconstitution of underlying index"` (`qid: 6410`, BM25/TF-IDF/Jaccard $\text{AP} = 1.0$ vs. Embeddings $\text{AP} = 0.0$).
   * *Causa:* Cuando el término exacto (`pre-market`, `reconstitution`) tiene un peso IDF muy alto en el índice invertido, BM25 aísla inmediatamente el documento correcto; en cambio, el modelo semántico prioriza documentos que hablan de horarios de bolsa o comportamiento general de ETFs sin responder al mecanismo puntual consultado.